This notebook creates precipitation time series from the MSWEP dataset. It uses the basin shapefile to extract precipitation over each basin and then computes the spatial mean to obtain daily precipitation time series.

In [1]:
import os
import xarray as xr
import geopandas as gpd
import rioxarray
import glob
import pandas as pd
from pathlib import Path
from tqdm import tqdm
import tempfile
from exactextract import exact_extract
from concurrent.futures import ThreadPoolExecutor, as_completed

In [ ]:
#Hides line: getfattr: /inputs/MSWEP_V280/Past/Daily/1989001.nc: Operation not supported

import os, sys 
sys.stderr = open(os.devnull, "w");

getfattr: /inputs/MSWEP_V280/Past/Daily/1989001.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989003.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989004.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989002.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989005.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989006.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989008.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989007.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989009.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989010.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989011.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989012.nc: Operation not supported
getfattr: /inputs/MSWEP_V280/Past/Daily/1989013.nc: Operation not supported
getfattr: /i

In [3]:
# ── Inputs ──────────────────────────────────────────────────────────────────
path_data = '/inputs/MSWEP_V280/Past/Daily'
path_shapefiles = Path('./shapefiles')

# ── Outputs ─────────────────────────────────────────────────────────────────
output_dir = Path("./precip_timeseries_mswep_weighted")
output_dir.mkdir(exist_ok=True)

In [4]:
# List all files
all_files = sorted(glob.glob(f'{path_data}/*.nc'))

In [5]:
# Filter by year 1989–2008
# files = [f for f in all_files if 1989 <= int(os.path.basename(f)[:4]) <= 2008]
files = [f for f in all_files if 1989 <= int(os.path.basename(f)[:4]) <= 2019]

print(f"Total files to process: {len(files)}")

Total files to process: 11322


In [6]:
basins_mapping = {
    'paso_mazangano': 'CAMELS_UY_10',
    'picada_de_coelho': 'CAMELS_UY_7',
    'sarandi_del_yi': 'CAMELS_UY_12',
    'paso_de_las_toscas': 'CAMELS_UY_8',
    'paso_de_las_piedras_rn': 'CAMELS_UY_15',
    'paso_del_borracho': 'CAMELS_UY_6',
    'bequelo': 'CAMELS_UY_16',
    'paso_de_las_piedras': 'CAMELS_UY_2',
    'paso_baltasar': 'CAMELS_UY_5',
    'fraile_muerto': 'CAMELS_UY_11',
    'paso_de_los_mellizos': 'CAMELS_UY_14',
    'paso_manuel_diaz': 'CAMELS_UY_3',
    'paso_aguiar': 'CAMELS_UY_9',
    'paso_de_la_compania': 'CAMELS_UY_1',
    'tacuarembo': 'CAMELS_UY_4',
    'durazno': 'CAMELS_UY_13'
}

In [7]:
shapefiles = os.listdir(path_shapefiles)
shapefiles


['durazno.zip',
 'paso_de_las_piedras.zip',
 'paso_de_la_compania.zip',
 'fraile_muerto.zip',
 'paso_aguiar.zip',
 'mercedes.zip',
 'bequelo.zip',
 'sarandi_del_yi.zip',
 'paso_manuel_diaz.zip',
 'paso_de_los_mellizos.zip',
 'paso_de_las_toscas.zip',
 'paso_del_borracho.zip',
 'picada_de_coelho.zip',
 'paso_mazangano.zip',
 'paso_de_las_piedras_rn.zip',
 'paso_baltasar.zip',
 'tacuarembo.zip']

In [10]:
def process_file(f, gdf_all):
    ds = xr.open_dataset(f)
    da = ds["precipitation"].squeeze()

    da = da.rename({"lat": "y", "lon": "x"})
    da = da.rio.write_crs("EPSG:4326")

    with tempfile.NamedTemporaryFile(suffix=".tif", delete=False) as tmp:
        tmp_path = tmp.name
    try:
        da.rio.to_raster(tmp_path)
        result = exact_extract(tmp_path, gdf_all, ["mean"], output="pandas")
        values = dict(zip(gdf_all["basin"], result["mean"].values))
    finally:
        os.unlink(tmp_path)

    date = pd.to_datetime(ds.time.values[0])
    ds.close()
    return {"date": date, **values}

In [11]:
# Load all shapefiles into a single GeoDataFrame
gdfs = []
for shapefile in shapefiles:
    gdf = gpd.read_file(path_shapefiles / shapefile)
    if gdf.crs != "EPSG:4326":
        gdf = gdf.to_crs("EPSG:4326")
    basin_key = os.path.splitext(shapefile)[0].lower().replace(" ", "_")
    gdf["basin"] = basins_mapping.get(basin_key, basin_key)
    gdfs.append(gdf)

gdf_all = gpd.GeoDataFrame(pd.concat(gdfs, ignore_index=True), crs="EPSG:4326")
basins = list(gdf_all["basin"])

In [14]:
all_results = []

with ThreadPoolExecutor(max_workers=4) as executor:
    futures = {executor.submit(process_file, f, gdf_all): f for f in files}
    for future in tqdm(as_completed(futures), total=len(files),
                       desc="Processing files", file=sys.stdout):
        all_results.append(future.result())

# Save one CSV per basin
for basin in basins:
    ts = pd.DataFrame(
        [{"date": r["date"], "precip_mm": r[basin]} for r in all_results]
    ).sort_values("date").set_index("date")
    ts.to_csv(output_dir / f"{basin}_precip.csv")
    print(f"Saved: {basin}")

Processing files: 100%|██████████| 11322/11322 [3:40:50<00:00,  1.17s/it]  
Saved: CAMELS_UY_13
Saved: CAMELS_UY_2
Saved: CAMELS_UY_1
Saved: CAMELS_UY_11
Saved: CAMELS_UY_9
Saved: mercedes
Saved: CAMELS_UY_16
Saved: CAMELS_UY_12
Saved: CAMELS_UY_3
Saved: CAMELS_UY_14
Saved: CAMELS_UY_8
Saved: CAMELS_UY_6
Saved: CAMELS_UY_7
Saved: CAMELS_UY_10
Saved: CAMELS_UY_15
Saved: CAMELS_UY_5
Saved: CAMELS_UY_4
